Libraries and functions requiered for training

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
from UNet import UNet
from dataloader import SolarPanelSegmentationDataset

Transformation, loading data and batch size definition

In [ ]:
# Define transformation for preprocessing
transform = transforms.Compose([
    #transforms.RandomHorizontalFlip(),
    #transforms.RandomVerticalFlip(),
    transforms.ToTensor()
])

# Define dataset directories
image_dir = r'C:\Users\32esla1mpg\Documents\UNet_PV\train_data\images'
mask_dir = r'C:\Users\32esla1mpg\Documents\UNet_PV\train_data\masks'

# Create a single dataset for images and masks
dataset = SolarPanelSegmentationDataset(image_dir, mask_dir, transform=transform)

print("dataset size", len(dataset))

batch_size = 40  # 40,  Adjust batch size based on available memory and dataset size
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# Initialize U-Net model
in_channels = 3  # RGB images
out_channels = 1  # Binary segmentation (solar panels or background)
model = UNet(in_channels, out_channels)

# Training loop
num_epochs = 200

# Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01) #0.0001
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, len(dataset)/batch_size*num_epochs) #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.train()

Calculation of metrics (accuracy, dice, IoU)

In [ ]:

# Metrics function
def calculate_metrics(predictions, targets, epsilon=1e-6):
    predictions = (predictions > 0.5).float()
    targets = targets.float()
    
    intersection = torch.logical_and(predictions, targets).sum().float()
    union = torch.logical_or(predictions, targets).sum().float()
    iou = intersection / (union + epsilon)

    dice = (2 * intersection) / (predictions.sum().float() + targets.sum().float() + epsilon)

    accuracy = (predictions == targets).sum().float() / torch.numel(targets)

    return iou.item(), dice.item(), accuracy.item()

Training loop over dataset and loss calculation

In [ ]:
# Initialize an empty list to store training losses for each epoch
running_loss = []

# Training loop
log_file_path = r'C:\Users\32esla1mpg\Documents\UNet_PV\outputs\training_metrics.txt'
with open(log_file_path, 'w') as f:
    f.write("Epoch\tLoss\tAccuracy\tIoU\tDice\n")  # Write the header
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        total_iou, total_dice, total_accuracy = 0.0, 0.0, 0.0
        
        for batch_idx, (inputs, masks, filenames) in enumerate(dataloader):
            inputs = inputs.to(device)
            masks = masks.to(device)
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)

            # Calculate loss
            loss = criterion(outputs, masks)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # Convert outputs to binary predictions
            predictions = (outputs > 0.5).float()

            # Calculate metrics
            iou, dice, accuracy = calculate_metrics(predictions, masks)
            
            # Accumulate metrics
            total_iou += iou
            total_dice += dice
            total_accuracy += accuracy

        # Average metrics over all batches
        average_iou = total_iou / len(dataloader)
        average_dice = total_dice / len(dataloader)
        average_accuracy = total_accuracy / len(dataloader)

        log_entry = f"{epoch+1}\t{running_loss/len(dataloader):.4f}\t{average_accuracy:.4f}\t{average_iou:.4f}\t{average_dice:.4f}\n"
        f.write(log_entry)
    
        # Print epoch loss and metrics
        print(f'Epoch {epoch+1} - Loss: {running_loss/len(dataloader):.4f}, Accuracy: {average_accuracy:.4f}, IoU: {average_iou:.4f}, Dice: {average_dice:.4f}')

# Save trained model parameters
torch.save(model.state_dict(), r'C:\Users\32esla1mpg\Documents\UNet_PV\outputs\trained_PV.pth')

print('Training finished.')

Extra: Checking how the model is seening the masks after transformation.

In [ ]:
import matplotlib.pyplot as plt

sample = dataset[1][0].unsqueeze(0)
sample = sample.to(device)

label = dataset[1][1].numpy()

output = model(sample)
pred = output.detach().cpu().numpy() > 0.5

#plt.imshow(label[0], vmin=0, vmax=1)
plt.imshow(pred[0][0], vmin=0, vmax=1)